# Συνοδευτικό σημειωματάριο του κεφαλαίου 6
**Φτιάξε το πρώτο σου μεγάλο γλωσσικό μοντέλο — Κεφάλαιο 6: Οδηγός επιβίωσης για NumPy και PyTorch**

Εκτέλεσε τα κελιά με τη σειρά, από πάνω προς τα κάτω, για να δεις τα βασικά για τους τανυστές, την εφαρμογή μάσκας, την αυτόματη επέκταση διαστάσεων (broadcasting) και έναν μικρό βρόχο εκπαίδευσης.

**Η διαδρομή σου (ακολουθεί τη βασική πορεία του βιβλίου):**
1. **Προετοιμασία**: το κελί ακριβώς από κάτω, όπου εισάγεις τις βιβλιοθήκες.
2. **Αριθμητικά παραδείγματα**: πώς δημιουργείς τανυστές, ορίζεις σπόρους για τις γεννήτριες τυχαίων αριθμών, προσθέτεις διαστάσεις, αλλάζεις σχήμα, επιλέγεις στοιχεία με δείκτες, χρησιμοποιείς το broadcasting και εκτελείς βασικές πράξεις. Αυτό που αξίζει να σου μείνει είναι η ευχέρεια στον χειρισμό των τανυστών.
3. **Μια πρώτη ματιά (ρίξε μια ματιά τώρα και ξαναδιάβασε αργότερα)**: ο μηχανισμός προσοχής βήμα προς βήμα (Κεφάλαιο 10) και η μονάδα διανυσματικής αναπαράστασης (Κεφάλαιο 9).
4. **Έλεγχος και αποτέλεσμα**: ξεκίνα με το κελί «Ελάχιστος βρόχος εκπαίδευσης» (η συνταγή του βιβλίου σε 5 βήματα). Έπειτα, στο «Δες την απώλεια να μειώνεται», ένα μοντέλο με ένα βάρος μαθαίνει τη σχέση `y = 2x + 1`.

Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch06.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch06.ipynb). Οι αγγλικές ετικέτες στις μετρήσεις της απώλειας μένουν αυτούσιες για να συγκρίνεις την έξοδο με το βιβλίο.

In [ ]:
# ===== ΕΙΣΑΓΩΓΗ ΒΙΒΛΙΟΘΗΚΩΝ =====
# torch: Η βιβλιοθήκη PyTorch για πράξεις με τανυστές (σκέψου τους ως πολυδιάστατους πίνακες)
# numpy: Η κλασική βιβλιοθήκη αριθμητικών υπολογισμών (από αυτήν εμπνεύστηκε το PyTorch)
# torch.nn: Δομικά στοιχεία νευρωνικών δικτύων (επίπεδα κ.λπ.)
import torch, numpy as np
import torch.nn as nn

print('Έκδοση PyTorch:', torch.__version__)
print('Διαθεσιμότητα CUDA:', torch.cuda.is_available())
# CUDA = τεχνολογία για υπολογισμούς σε GPU. Αν δεις True, μπορείς να χρησιμοποιήσεις τη GPU για επιτάχυνση (10–100× στη βαθιά μάθηση)

## Τι είναι ο τανυστής;

Ένας **τανυστής** είναι ένας πολυδιάστατος πίνακας αριθμών:
- **Μονοδιάστατος τανυστής** = μια λίστα `[1, 2, 3]` (σαν γραμμή σε ένα υπολογιστικό φύλλο)
- **Δισδιάστατος τανυστής** = ένας πίνακας με γραμμές και στήλες
- **Τρισδιάστατος τανυστής** = μια στοίβα από πίνακες (σαν πολλά φύλλα στο Excel)

**Γιατί να προτιμήσεις το PyTorch αντί για το NumPy;**
Και τα δύο χειρίζονται πολυδιάστατους πίνακες, αλλά το PyTorch σου προσφέρει επιπλέον:
1. **Υποστήριξη GPU** — μετάφερε δεδομένα στη GPU για επιτάχυνση κατά 10–100 φορές
2. **Αυτόματο υπολογισμό κλίσεων** — υπολογίζει τις παραγώγους που χρειάζεσαι για την εκπαίδευση νευρωνικών δικτύων
3. **Επίπεδα νευρωνικών δικτύων** — έτοιμα δομικά στοιχεία

## Δημιουργία τανυστών
Δημιούργησε τανυστές από δεδομένα Python ή NumPy, αλλά και με συνηθισμένους κανόνες συμπλήρωσης τιμών.

In [ ]:
# ===== Δημιουργία τανυστών =====

# Από δεδομένα Python
data = torch.tensor([[1, 2, 3], [4, 5, 6]])  # Τανυστής 2×3

# Δημιούργησε τανυστές με σταθερές ή τυχαίες τιμές
zeros = torch.zeros(3, 4)     # Τανυστής 3×4 με μηδενικά
ones = torch.ones(2, 3, 4)    # Τανυστής 2×3×4 με μονάδες
uniform = torch.rand(3, 4)    # Τυχαίες τιμές από ομοιόμορφη κατανομή στο [0, 1)
normal = torch.randn(3, 4)    # Τυχαίες τιμές από κανονική κατανομή (μέση τιμή 0, τυπική απόκλιση 1)
integers = torch.randint(0, 10, (3, 4))  # Τυχαίοι ακέραιοι στο [0, 10)

# Αριθμητικές ακολουθίες (όπως με τη range της Python)
sequence = torch.arange(0, 10, 2)  # [0, 2, 4, 6, 8]
linspace = torch.linspace(0, 1, 5)  # 5 ισαπέχοντα σημεία από το 0 έως το 1

# Τύποι δεδομένων (dtype) — όρισε την ακρίβεια
float_tensor = torch.tensor([1.0, 2.0], dtype=torch.float32)  # Αριθμοί κινητής υποδιαστολής 32 bit (προεπιλογή)
int_tensor = torch.tensor([1, 2], dtype=torch.long)           # Ακέραιοι 64 bit (για δείκτες)

# NumPy ↔ PyTorch (ο πίνακας και ο τανυστής μοιράζονται τη μνήμη: οι αλλαγές στον έναν φαίνονται και στον άλλο!)
np_data = np.array([1, 2, 3], dtype=np.float32)
torch_from_np = torch.from_numpy(np_data)  # Μοιράζεται τη μνήμη με τον πίνακα np_data
back_to_np = torch_from_np.numpy()         # Επιστροφή σε πίνακα NumPy

# Επιλογή συσκευής — CPU ή GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_tensor = torch.randn(3, 4, device=device, dtype=torch.float32)

print('Σχήμα του zeros:', zeros.shape)
print('Συσκευή του gpu_tensor:', gpu_tensor.device)

## Αναπαραγωγιμότητα: Ορισμός σπόρου τυχαιότητας
Απαραίτητο βήμα για να εντοπίζεις σφάλματα και να συγκρίνεις πειράματα

In [ ]:
# Όρισε σταθερό σπόρο τυχαιότητας για να αναπαράγεις τα αποτελέσματα
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Σε κάθε εκτέλεση, ο a θα έχει τις ίδιες τιμές με πριν, όπως και ο b
a = torch.randn(3, 4)
b = torch.randn(3, 4)

# Δοκίμασε χωρίς να ορίσεις σπόρο: τα αποτελέσματα αλλάζουν κάθε φορά
# Αν όμως ορίσεις τον ίδιο σπόρο, μπορείς να τα αναπαράγεις
torch.manual_seed(42)
x1 = torch.randn(2, 3)
torch.manual_seed(42)
x2 = torch.randn(2, 3)
print('Με τον ίδιο σπόρο, οι τανυστές έχουν τις ίδιες τιμές:', torch.allclose(x1, x2))

# Γιατί έχει σημασία: μπορείς να αναπαράγεις τα πειράματά σου τόσο για τον εντοπισμό σφαλμάτων όσο και για την έρευνα

## Από τις λίστες στους τανυστές: προσθέτοντας διαστάσεις
Σύνδεσε τις απλές λίστες του Κεφαλαίου 5 με τους πολυδιάστατους τανυστές.

In [ ]:
# Βήμα 1: Μονοδιάστατοι τανυστές (παράδειγμα από το Κεφάλαιο 5)
# Από το Κεφάλαιο 5: τα αναγνωριστικά των token που δίνει ο tokenizer
token_ids = [2, 3, 4, 6]
tokens_1d = torch.tensor(token_ids)
print(f"Σχήμα μονοδιάστατου τανυστή: {tokens_1d.shape}")  # torch.Size([4])
print(f"Δεδομένα: {tokens_1d}")

In [ ]:
# Βήμα 2: Ομαδοποίηση σε παρτίδες (από μία σε δύο διαστάσεις)
# Επεξεργάσου πολλές προτάσεις ταυτόχρονα
batch = torch.tensor([
    [2, 3, 4, 6],    # πρόταση 1
    [5, 7, 8, 9]     # πρόταση 2
])
print(f"Σχήμα δισδιάστατης παρτίδας: {batch.shape}")  # torch.Size([2, 4])
print("Πρώτη πρόταση:", batch[0])
print("Δεύτερο token της πρώτης πρότασης:", batch[0, 1])

In [ ]:
# Βήμα 3: Διανυσματικές αναπαραστάσεις (2D → 3D)
# Πρόσθεσε 768 αριθμούς ανά token (όπως στο GPT-2)
embeddings_3d = torch.randn(2, 4, 768)
print(f"Σχήμα του τρισδιάστατου τανυστή διανυσματικών αναπαραστάσεων: {embeddings_3d.shape}")  # torch.Size([2, 4, 768])

# Διάσχισε τις διαστάσεις με τη σειρά: παρτίδα → token → χαρακτηριστικά
print("Διανυσματικές αναπαραστάσεις της πρώτης πρότασης:", embeddings_3d[0].shape)      # (4, 768)
print("Πρώτο token της πρώτης πρότασης:", embeddings_3d[0, 0].shape)  # (768,)

In [ ]:
# Βήμα 4: Μια πρώτη ματιά στις κεφαλές προσοχής (3D → 4D)
# Η προσοχή με πολλαπλές κεφαλές προσθέτει άλλη μία διάσταση (μη σε απασχολούν ακόμη οι λεπτομέρειες)
attention_4d = torch.randn(2, 8, 4, 4)  # (παρτίδα, κεφαλές, μήκος ακολουθίας, μήκος ακολουθίας)
print(f"Σχήμα του τετραδιάστατου τανυστή προσοχής: {attention_4d.shape}")
print("Ερμηνεία του σχήματος: (παρτίδα, κεφαλές, μήκος ακολουθίας, μήκος ακολουθίας)")

## Αλλαγή σχήματος, αφαίρεση διαστάσεων μεγέθους 1 και αναδιάταξη διαστάσεων
Αυτές τις τεχνικές θα τις χρησιμοποιείς συνέχεια για να αλλάζεις το σχήμα ενός τανυστή.

In [ ]:
# ===== Πράξεις αναδιαμόρφωσης =====
# Αναδιαμόρφωση σημαίνει ότι αναδιοργανώνεις τα δεδομένα χωρίς να αλλάζεις τις τιμές τους (σαν να μετακινείς κουτιά σε μια αποθήκη)

x = torch.arange(12)               # Δημιούργησε έναν μονοδιάστατο τανυστή [0,1,2,...,11] με σχήμα (12,)
print(f'Αρχικό σχήμα: {x.shape}')

# view() — δίνει νέα όψη χωρίς αντιγραφή, αν η διάταξη των στοιχείων στη μνήμη είναι συμβατή με το νέο σχήμα
x = x.view(3, 4)                   # Άλλαξε το σχήμα σε (3, 4): 3 γραμμές × 4 στήλες
print(f'Μετά τη view(3,4): {x.shape}')

# reshape() — μοιάζει με τη view(), αλλά χειρίζεται και τανυστές που δεν καταλαμβάνουν συνεχόμενες θέσεις μνήμης (προτίμησέ τη αν έχεις αμφιβολίες)
x = x.reshape(2, 2, 3)             # Άλλαξε το σχήμα σε (2, 2, 3)
print(f'Μετά τη reshape(2,2,3): {x.shape}')

# Το -1 σημαίνει «υπολόγισε αυτόματα το μέγεθος αυτής της διάστασης»
x = x.view(-1, 3)                  # Το -1 γίνεται 4 (12 στοιχεία συνολικά ÷ 3 = 4)
print(f'Μετά τη view(-1,3): {x.shape}')

# unsqueeze/squeeze — πρόσθεσε ή αφαίρεσε διαστάσεις μεγέθους 1
y = torch.randn(3, 4)
y = y.unsqueeze(0)                 # (1, 3, 4) — πρόσθεσε τη διάσταση παρτίδας στη θέση 0
y = y.unsqueeze(-1)                # (1, 3, 4, 1) — πρόσθεσε μια διάσταση στο τέλος
y = y.squeeze()                    # Αφαίρεσε ΟΛΕΣ τις διαστάσεις μεγέθους 1 → (3, 4)
print(f'Μετά τη squeeze: {y.shape}')

# permute — άλλαξε τη σειρά των διαστάσεων (όπως στη μετάθεση ενός πίνακα, αλλά για οποιοδήποτε πλήθος διαστάσεων)
z = torch.randn(2, 3, 4)           # (παρτίδα, ακολουθία, χαρακτηριστικά)
z = z.permute(0, 2, 1)             # (παρτίδα, χαρακτηριστικά, ακολουθία) — αντάλλαξε τις δύο τελευταίες διαστάσεις
print(f'Μετά την permute: {z.shape}')

# flatten — συγχώνευσε διαστάσεις σε μία
t = torch.randn(2, 3, 4)
t_flat = t.flatten()               # (24,) — συγχώνευσε όλες τις διαστάσεις σε μία
t_flat_features = t.flatten(1)     # (2, 12) — συγχώνευσε τις διαστάσεις από τη διάσταση 1 και μετά
print(f'Πλήρης επιπέδωση: {t_flat.shape}')
print(f'Επιπέδωση χαρακτηριστικών: {t_flat_features.shape}')

## Επιλογή με δείκτες και τομές
Επίλεξε παρτίδες και token και χρησιμοποίησε λογικές μάσκες.

In [ ]:
# ===== Επιλογή με δείκτες και τομές =====
# Περιηγήσου στα πολυδιάστατα δεδομένα όπως θα περιηγούσουν σε φακέλους: παρτίδα → token → χαρακτηριστικά

# Δημιούργησε έναν τετραδιάστατο τανυστή που προσομοιώνει την προσοχή: (παρτίδα, κεφαλές, ακολουθία, ακολουθία)
x = torch.randn(2, 4, 6, 6)  # 2 παρτίδες, 4 κεφαλές, 6 token, 6 token

# Απλή επιλογή με δείκτες
first_batch = x[0]              # Σχήμα: (4, 6, 6) — πρώτη παρτίδα, όλες οι κεφαλές
first_head = x[0, 0]            # Σχήμα: (6, 6) — πρώτη παρτίδα, πρώτη κεφαλή
single_value = x[0, 0, 0, 0]    # Σχήμα: () — ένας μόνο αριθμός

# Τομές με το σύμβολο :
first_two_batches = x[:2]       # Σχήμα: (2, 4, 6, 6) — οι 2 πρώτες παρτίδες
all_but_last_token = x[:, :, :-1, :]  # Σχήμα: (2, 4, 5, 6) — παράλειψε το τελευταίο token
every_other_head = x[:, ::2]    # Σχήμα: (2, 2, 6, 6) — οι κεφαλές 0 και 2

# Οι αρνητικοί δείκτες μετρούν από το τέλος
last_token = x[:, :, -1, :]     # Σχήμα: (2, 4, 6) — το τελευταίο token κάθε ακολουθίας

# Επιλογή με λογική μάσκα (φιλτράρισμα βάσει συνθήκης)
mask = torch.tensor([True, False, True, False])
filtered_heads = x[0, mask]     # Σχήμα: (2, 6, 6) — μόνο οι κεφαλές 0 και 2

print(f'Αρχικό σχήμα: {x.shape}')
print(f'Σχήμα πρώτης παρτίδας: {first_batch.shape}')
print(f'Χωρίς το τελευταίο token: {all_but_last_token.shape}')
print(f'Επιλεγμένες κεφαλές: {filtered_heads.shape}')

## Προσοχή: η καρδιά των Transformer

**Η βασική ιδέα:** Η προσοχή υπολογίζει έναν σταθμισμένο μέσο όρο για όλες τις λέξεις, με βάρη που προκύπτουν από βαθμολογίες συνάφειας. Σκέψου πώς διαβάζεις την πρόταση «Η γάτα κάθισε στο χαλάκι»: για να καταλάβεις το «κάθισε», στρέφεις την προσοχή σου στη «γάτα» (ποιος κάθισε;) και στο «χαλάκι» (πού κάθισε;).

Θα υλοποιήσεις την προσοχή σε 3 βήματα:
1. Κάνε τον βασικό υπολογισμό (4 πράξεις)
2. Πρόσθεσε αιτιακή μάσκα (για να μην έχεις πρόσβαση σε μελλοντικά token)
3. Χρησιμοποίησε την έτοιμη λύση για περιβάλλον παραγωγής (το PyTorch αναλαμβάνει όλο τον υπολογισμό)

In [ ]:
import torch
import torch.nn.functional as F

# ===== Τα βασικά μαθηματικά της προσοχής =====
# Προσοχή: ερώτημα (τι ψάχνω;), κλειδί (τι διαθέτω;), τιμή (τι να επιστρέψω;)

# Προσομοίωσε διανυσματικές αναπαραστάσεις για μια παρτίδα 2 ακολουθιών με 5 token η καθεμία
batch, seq_len, d_head = 2, 5, 64
Q = torch.randn(batch, seq_len, d_head)  # (2, 5, 64)
K = torch.randn(batch, seq_len, d_head)
V = torch.randn(batch, seq_len, d_head)

# Βήμα 1: Υπολόγισε τις βαθμολογίες (πόσο ταιριάζει κάθε token με κάθε token της ακολουθίας;)
# Ο τελεστής @ κάνει πολλαπλασιασμό πινάκων (όπως και η torch.matmul)
# Η K.transpose(-2, -1) ανταλλάσσει τις δύο τελευταίες διαστάσεις: (2, 5, 64) → (2, 64, 5)
# Αποτέλεσμα: (2, 5, 64) @ (2, 64, 5) → (2, 5, 5) — ένας πίνακας βαθμολογιών 5×5 για κάθε ακολουθία της παρτίδας
scores = Q @ K.transpose(-2, -1)
print(f'Σχήμα του τανυστή βαθμολογιών: {scores.shape}')  # (2, 5, 5)

# Βήμα 2: Άλλαξε την κλίμακα των βαθμολογιών (για να αποφύγεις τον κορεσμό της softmax)
# Χωρίς κλιμάκωση, τα μεγάλα εσωτερικά γινόμενα οδηγούν τη softmax να δίνει πιθανότητα ≈1 στη μέγιστη βαθμολογία και ≈0 στις υπόλοιπες
scores = scores / (d_head ** 0.5)  # διαίρεσε με sqrt(64) = 8

# Βήμα 3: Εφάρμοσε τη softmax (μετέτρεψε τις βαθμολογίες σε πιθανότητες)
# Το dim=-1 σημαίνει «κατά μήκος της τελευταίας διάστασης» (της διάστασης των κλειδιών)
attn_weights = torch.softmax(scores, dim=-1)  # το άθροισμα κάθε γραμμής είναι 1

# Βήμα 4: Υπολόγισε το σταθμισμένο άθροισμα (συνδύασε τις τιμές με βάρη τις πιθανότητες)
output = attn_weights @ V  # (2, 5, 64)

print(f'Σχήμα της εξόδου: {output.shape}')  # ίδιο με το σχήμα της εισόδου: (2, 5, 64)
print(f'Το άθροισμα των βαρών προσοχής είναι 1: {attn_weights[0, 0].sum():.4f}')
print(f'\nΒάρη προσοχής του token στη θέση 2: {attn_weights[0, 2]}')
print('(δείχνουν πόση προσοχή δίνει το token στη θέση 2 σε καθένα από τα 5 token)')

## Διανυσματικές αναπαραστάσεις των token και των θέσεών τους

Κάθε μεγάλο γλωσσικό μοντέλο (LLM) ξεκινά μετατρέποντας τα αναγνωριστικά των token σε πυκνά διανύσματα.

**Το πρόβλημα:** Τα νευρωνικά δίκτυα δεν μπορούν να επεξεργαστούν απευθείας κείμενο όπως η λέξη «γάτα». Τα αναγνωριστικά των token (όπως το `5` για τη «γάτα») είναι αυθαίρετα: το αναγνωριστικό 5 δεν είναι «πιο κοντά» στο 6 απ’ ό,τι στο 500.

**Η λύση:** Αντιστοίχισε κάθε αναγνωριστικό token σε μια διανυσματική αναπαράσταση που μαθαίνει το μοντέλο (768 αριθμοί στο GPT-2). Κατά την εκπαίδευση, λέξεις με παρόμοια σημασία αποκτούν παρόμοιες διανυσματικές αναπαραστάσεις.

In [ ]:
# ===== Διανυσματικές αναπαραστάσεις token και θέσεων =====
# Διαστάσεις του GPT-2
vocab_size, d_model, max_seq_len = 50257, 768, 1024

# Διανυσματικές αναπαραστάσεις token: ένας πίνακας αναζήτησης (σαν λεξικό: token_id → διάνυσμα)
# Η nn.Embedding δημιουργεί έναν πίνακα με vocab_size γραμμές και d_model στήλες
# Κάθε γραμμή είναι ένα διάνυσμα 768 διαστάσεων που αναπαριστά ένα token
token_embedding = nn.Embedding(vocab_size, d_model)

# Δημιούργησε τυχαία αναγνωριστικά token (φαντάσου ότι τα έδωσε ένας tokenizer)
token_ids = torch.randint(0, vocab_size, (2, 5))  # 2 προτάσεις με 5 token η καθεμία

# Βρες τις διανυσματικές αναπαραστάσεις (τα αναγνωριστικά είναι απλώς δείκτες γραμμών!)
token_vectors = token_embedding(token_ids)
print(f'Διανυσματικές αναπαραστάσεις token: {token_vectors.shape}')  # (2, 5, 768)

# Διανυσματικές αναπαραστάσεις θέσεων: πού βρίσκεται κάθε token στην ακολουθία (θέση 0, θέση 1 κ.ο.κ.)
# Η ίδια ιδέα: ένας πίνακας αναζήτησης όπου η γραμμή i αναπαριστά τη «θέση i»
pos_embedding = nn.Embedding(max_seq_len, d_model)
position_ids = torch.arange(5).unsqueeze(0).expand(2, -1)  # [[0,1,2,3,4], [0,1,2,3,4]]
pos_vectors = pos_embedding(position_ids)
print(f'Διανυσματικές αναπαραστάσεις θέσεων: {pos_vectors.shape}')  # (2, 5, 768)

# Συνδύασέ τες με πρόσθεση κατά στοιχείο (ίδια θέση, ίδιο σχήμα!)
# Γιατί πρόσθεση αντί για συνένωση; Με την πρόσθεση η διάσταση παραμένει 768 (αντί για 1536)
# Το μοντέλο μαθαίνει να κωδικοποιεί ΚΑΙ τη σημασία ΚΑΙ τη θέση στο ίδιο διάνυσμα
input_embeddings = token_vectors + pos_vectors
print(f'Συνδυασμένες αναπαραστάσεις: {input_embeddings.shape}')  # (2, 5, 768)

# Μέτρηση παραμέτρων (πόσους αριθμούς πρέπει να μάθει το μοντέλο;)
token_params = vocab_size * d_model   # 50.257 token × 768 διαστάσεις
pos_params = max_seq_len * d_model    # 1.024 θέσεις × 768 διαστάσεις
print(f'Παράμετροι για τα token: {token_params:,}')      # 38,597,376
print(f'Παράμετροι για τις θέσεις: {pos_params:,}')     # 786,432
print(f'Σύνολο: {token_params + pos_params:,}')  # 39,383,808

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)  # όρισε τον σπόρο τυχαιότητας ΠΡΙΝ δημιουργήσεις το μοντέλο (όπως στο βιβλίο)

# Απλό MLP δύο επιπέδων για ένα αυτοτελές παράδειγμα ταξινόμησης.
# Είναι η ίδια κλάση όπως στην ενότητα «Δημιουργία επιπέδων με το nn.Module» του βιβλίου:
# fc1 -> ReLU -> LayerNorm -> fc2.
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)      # Γραμμικός μετασχηματισμός
        x = self.act(x)      # Μη γραμμική ενεργοποίηση (ReLU)
        x = self.norm(x)     # Κανονικοποίηση
        x = self.fc2(x)      # Προβολή εξόδου
        return x

# Η συνταγή εκπαίδευσης σε 5 βήματα (με την ίδια σειρά όπως στο βιβλίο):
# 1. Μηδένισε τις κλίσεις → καθάρισε τις παλιές κλίσεις (από προεπιλογή συσσωρεύονται!)
# 2. Κάνε το πρόσθιο πέρασμα → πάρε τις προβλέψεις
# 3. Υπολόγισε την απώλεια → μέτρησε το σφάλμα
# 4. Κάνε οπισθοδιάδοση → υπολόγισε τις κλίσεις
# 5. Ενημέρωσε τα βάρη → προσάρμοσε τις παραμέτρους
# Για μεγάλα γλωσσικά μοντέλα (LLM), πρόσθεσε περιορισμό κλίσεων ανάμεσα στα βήματα 4 και 5.

# Προετοιμασία: μοντέλο, βελτιστοποιητής, συνάρτηση απώλειας, τεχνητά δεδομένα (με σταθερό σπόρο, όπως στο βιβλίο)
model = SimpleMLP(16, 32, 2)  # είσοδος 16 διαστάσεων, κρυφό επίπεδο 32 διαστάσεων, έξοδος για 2 κλάσεις
optimizer = optim.Adam(model.parameters(), lr=1e-3)  # προσαρμοστικός ρυθμός μάθησης
loss_fn = nn.CrossEntropyLoss()  # για ταξινόμηση
inputs = torch.randn(64, 16)          # 64 παραδείγματα με 16 χαρακτηριστικά το καθένα
labels = torch.randint(0, 2, (64,))   # 64 ετικέτες (κλάση 0 ή 1)

# Εκτέλεσε τα 5 βήματα μία φορά (ο «Ελάχιστος βρόχος εκπαίδευσης» του βιβλίου)
optimizer.zero_grad(set_to_none=True)  # 1. καθάρισε τις παλιές κλίσεις (συσσωρεύονται!)
logits = model(inputs)                 # 2. κάνε το πρόσθιο πέρασμα (ακατέργαστες βαθμολογίες, όχι πιθανότητες)
loss = loss_fn(logits, labels)         # 3. μέτρησε το σφάλμα των προβλέψεων
print(f"before: loss={loss.item():.4f}")
loss.backward()                        # 4. αποθήκευσε την κλίση κάθε παραμέτρου στο πεδίο .grad
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # δικλίδα ασφαλείας για τα LLM
optimizer.step()                       # 5. προσάρμοσε τα βάρη με βάση τις κλίσεις

with torch.no_grad():
    after = loss_fn(model(inputs), labels)
print(f"after:  loss={after.item():.4f}")

print()
print("✅ Βασικά σημεία:")
print("  - Η .backward() αποθηκεύει την κλίση κάθε παραμέτρου στο πεδίο .grad")
print("  - Ο βελτιστοποιητής χρησιμοποιεί τις κλίσεις για να προσαρμόσει τα βάρη")
print("  - Ο περιορισμός κλίσεων θέτει ανώτατο όριο στη συνολική νόρμα των κλίσεων πριν από την ενημέρωση των βαρών")
print("  - Οι ετικέτες είναι τυχαίος θόρυβος και η αξιολόγηση γίνεται στα ίδια παραδείγματα")
print("    που χρησιμοποιήθηκαν για την εκπαίδευση. Άρα η ακρίβεια εδώ αφορά το δείγμα εκπαίδευσης,")
print("    όχι τη γενίκευση. Στο Κεφάλαιο 12 θα φτιάξεις τον πλήρη βρόχο εκπαίδευσης για πολλές εποχές.")


### Δες την απώλεια να μειώνεται (το αποτέλεσμα που υπόσχεται το βιβλίο)

Κάνε μια σύντομη αλλά πλήρη εκπαίδευση ακολουθώντας ακριβώς τη συνταγή των 5 βημάτων: μάθε σε ένα μοντέλο με ένα βάρος και έναν όρο μετατόπισης τον κανόνα `y = 2x + 1` και δες το σφάλμα να μειώνεται. Η άσκηση αντιστοιχεί στο σημείο ελέγχου «Δες την απώλεια να μειώνεται» του βιβλίου.

In [ ]:
import torch
torch.manual_seed(42)
x = torch.linspace(0, 1, 32).unsqueeze(1)  # 32 είσοδοι, σχήμα (32, 1)
y = 2 * x + 1                              # ο κανόνας που θέλουμε να μάθει το μοντέλο
model = torch.nn.Linear(1, 1)              # ένα βάρος και ένας όρος μετατόπισης
opt = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = torch.nn.MSELoss()
for step in range(1, 51):
    opt.zero_grad()                        # 1. μηδένισε τις παλιές κλίσεις
    loss = loss_fn(model(x), y)            # 2+3. κάνε το πρόσθιο πέρασμα και μέτρησε το σφάλμα
    loss.backward()                        # 4. υπολόγισε τις κλίσεις με οπισθοδιάδοση
    opt.step()                             # 5. ενημέρωσε τα βάρη
    if step % 10 == 0:
        print(f"step {step:2d}: loss = {loss.item():.6f}")
print(f"learned: y = {model.weight.item():.2f}x + {model.bias.item():.2f}")

In [ ]:
# Πλήρης μονάδα διανυσματικών αναπαραστάσεων (προεπισκόπηση της κλάσης του Κεφαλαίου 9)
class GPT2Embeddings(nn.Module):
    """Συνδυάζει τις διανυσματικές αναπαραστάσεις των token και των θέσεών τους όπως στο GPT-2 (θα ολοκληρώσεις την κλάση στο Κεφάλαιο 9)."""

    def __init__(self, vocab_size, max_seq_len, embed_dim):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        # Δημιούργησε τα αναγνωριστικά θέσης στην ΙΔΙΑ συσκευή με τα token για να αποφύγεις ένα συνηθισμένο σφάλμα
        position_ids = torch.arange(seq_len, device=token_ids.device)
        token_emb = self.token_embed(token_ids)     # (batch, seq, embed_dim)
        pos_emb = self.pos_embed(position_ids)      # (seq, embed_dim), επεκτείνεται αυτόματα σε όλη την παρτίδα κατά την πρόσθεση
        return token_emb + pos_emb

# Δοκίμασε τη μονάδα
embed_layer = GPT2Embeddings(vocab_size=50257, max_seq_len=1024, embed_dim=768)
token_ids = torch.randint(0, 50257, (2, 10))
output = embed_layer(token_ids)
print(f"Σχήμα εξόδου των διανυσματικών αναπαραστάσεων: {output.shape}")  # (2, 10, 768)

# Συνολικός αριθμός παραμέτρων
total_params = sum(p.numel() for p in embed_layer.parameters())
print(f"Σύνολο παραμέτρων των διανυσματικών αναπαραστάσεων: {total_params:,}")  # 39,383,808

Πρόσεξε ότι αυτή η κλάση δεν εφαρμόζει dropout: όταν συναρμολογήσεις το πλήρες μοντέλο στο Κεφάλαιο 11, το MiniGPT θα εφαρμόζει dropout μετά την πρόσθεση. Η ολοκληρωμένη έκδοση του Κεφαλαίου 9 ελέγχει επίσης το μήκος της ακολουθίας και αρχικοποιεί τα βάρη. Αυτή η προεπισκόπηση χρησιμοποιεί την ίδια υπογραφή και τα ίδια ονόματα με το Κεφάλαιο 9 (`embed_dim`, `token_embed`, `pos_embed`), ώστε να μπορείς να χρησιμοποιήσεις τον κώδικα χωρίς αλλαγές.

## Ο βρόχος εκπαίδευσης

**Βασικές έννοιες:**
- `nn.Module`: Βασική κλάση για τα επίπεδα ενός νευρωνικού δικτύου. Το μοντέλο σου κληρονομεί από αυτήν.
- `super().__init__()`: Καλεί τη μέθοδο αρχικοποίησης της γονικής κλάσης (απαραίτητο βήμα).
- `nn.Linear(in, out)`: Επίπεδο που εκτελεί πολλαπλασιασμό πινάκων (πίνακας βαρών in×out).
- `nn.ReLU()`: Συνάρτηση ενεργοποίησης — διατηρεί τις θετικές τιμές και μηδενίζει τις αρνητικές.
- `CrossEntropyLoss`: Συνάρτηση απώλειας που μετρά το σφάλμα των προβλέψεων ταξινόμησης.
- `optimizer.zero_grad()`: Μηδενίζει τις παλιές κλίσεις (από προεπιλογή συσσωρεύονται!).
- `.backward()`: Υπολογίζει τις κλίσεις με αυτόματη παραγώγιση κατά την οπισθοδιάδοση.
- `.step()`: Ενημερώνει τα βάρη με βάση τις κλίσεις που υπολογίστηκαν.

In [ ]:
# Βήμα 2: Πρόσθεσε αιτιακή μάσκα (για να μην κλέβει το μοντέλο)

# Πρόβλημα: Αν το token 2 βλέπει τα token 3 και 4, μπορεί να κλέψει κατά την εκπαίδευση!
# Λύση: Απόκλεισε τις μελλοντικές θέσεις ορίζοντας τη βαθμολογία τους σε -inf

# Πώς λειτουργεί η μάσκα (False = επιτρέπεται, True = αποκλείεται):
# Το token 0 βλέπει: [0]           ← μόνο τον εαυτό του
# Το token 1 βλέπει: [0, 1]        ← τα προηγούμενα token και τον εαυτό του
# Το token 2 βλέπει: [0, 1, 2]     ← τα προηγούμενα token και τον εαυτό του
# Το token 3 βλέπει: [0, 1, 2, 3]
# Το token 4 βλέπει: [0, 1, 2, 3, 4]

seq_len = 5
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

# Εφάρμοσε τη μάσκα πριν από τη softmax
scores = Q @ K.transpose(-2, -1) / (d_head ** 0.5)
scores = scores.masked_fill(causal_mask, float('-inf'))  # η softmax δίνει βάρος 0 στις θέσεις με βαθμολογία -inf
attn_weights = torch.softmax(scores, dim=-1)

print("Βάρη προσοχής με αιτιακή μάσκα (το token 2 βλέπει μόνο τα token 0, 1 και 2):")
print(attn_weights[0, 2])  # τα βάρη στις θέσεις 3 και 4 είναι μηδενικά
print("\nΤώρα το μοντέλο μαθαίνει να προβλέπει το 'mat' χωρίς να έχει δει πρώτα το 'mat'!")

In [ ]:
# Βήμα 3: Συντόμευση για κώδικα παραγωγής (σε μία γραμμή)

# Μόλις έμαθες να κάνεις τον υπολογισμό σε 4 βήματα, για να καταλάβεις τι συμβαίνει.
# Στην πράξη, το PyTorch αναλαμβάνει όλα τα βήματα για σένα:

output = F.scaled_dot_product_attention(
    Q, K, V,
    is_causal=True  # εφαρμόζει αυτόματα αιτιακή μάσκα
)

print(f"Σχήμα εξόδου: {output.shape}")  # (2, 5, 64)

# Γιατί να προτιμήσεις αυτή τη συνάρτηση από τον υπολογισμό βήμα προς βήμα;
# - Είναι 2–4 φορές πιο γρήγορη (χρησιμοποιεί FlashAttention)
# - Απαιτεί λιγότερη μνήμη (δεν αποθηκεύει ολόκληρο τον πίνακα προσοχής)
# - Χειρίζεται ειδικές περιπτώσεις (αριθμητική σταθερότητα, dropout, αυτόματη επέκταση της μάσκας στις κατάλληλες διαστάσεις)

print("\n✅ Πότε να κάνεις τον υπολογισμό βήμα προς βήμα και πότε να χρησιμοποιείς την έτοιμη συνάρτηση:")
print("   Για εξάσκηση: Γράψε τον υπολογισμό βήμα προς βήμα για να καταλάβεις τα μαθηματικά")
print("   Σε κώδικα παραγωγής: Χρησιμοποίησε πάντα τη F.scaled_dot_product_attention")

In [ ]:
seq_len = 5
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
print(mask)


## Παραδείγματα αυτόματης επέκτασης διαστάσεων
Πρόσθεση όρου μετατόπισης και εφαρμογή μάσκας με αυτόματη επέκταση διαστάσεων.

In [ ]:
x = torch.randn(3, 4)
y = x + 5
batch = torch.randn(32, 10, 768)
bias = torch.randn(768)
result = batch + bias  # Το bias επεκτείνεται αυτόματα στις δύο πρώτες διαστάσεις
scores = torch.randn(4, 8, 10, 10)
mask = torch.triu(torch.ones(1, 1, 10, 10), 1)
masked = scores + mask * -1e9
print('Σχήμα αποτελέσματος:', result.shape)


## Βασικές πράξεις: πράξεις ανά στοιχείο, πολλαπλασιασμός πινάκων, αναγωγές

**Οι πράξεις ανά στοιχείο** γίνονται θέση προς θέση, όπως όταν προσθέτεις τα αντίστοιχα κελιά δύο υπολογιστικών φύλλων. Αν οι τανυστές `a` και `b` έχουν και οι δύο σχήμα 3×4, τότε με την πράξη `a + b` προσθέτεις το `a[0,0]` στο `b[0,0]`, το `a[0,1]` στο `b[0,1]` και ούτω καθεξής. Το αποτέλεσμα έχει το ίδιο σχήμα με τις εισόδους.

In [ ]:
a = torch.randn(3, 4)
b = torch.randn(3, 4)
add = a + b
mul = a * b
square = a ** 2
exp = torch.exp(a)
x = torch.randn(32, 10, 64)
W = torch.randn(64, 128)
y = x @ W
total = a.sum()
row_sums = a.sum(dim=1, keepdim=True)
col_means = a.mean(dim=0)
max_vals, max_idx = a.max(dim=1)
c = torch.cat([a, b], dim=0)
d = torch.stack([a, b], dim=0)
logits = torch.randn(3, 5)
probs = torch.softmax(logits, dim=-1)
print('Σχήμα του y:', y.shape)
print('Αθροίσματα πιθανοτήτων ανά γραμμή:', probs.sum(dim=-1))


## Autograd: Αυτόματος υπολογισμός κλίσεων

**Τι είναι η κλίση;** Για μία μεταβλητή, είναι η παράγωγος: δείχνει πόσο γρήγορα αλλάζει η έξοδος όταν αλλάζεις την είσοδο.

**Γιατί σε ενδιαφέρει;** Όταν εκπαιδεύεις ένα νευρωνικό δίκτυο, προσαρμόζεις εκατομμύρια παραμέτρους. Οι κλίσεις σού δείχνουν προς ποια κατεύθυνση να τις προσαρμόσεις. Το autograd του PyTorch υπολογίζει τις κλίσεις αυτόματα.

In [ ]:
# Απλός γράφος υπολογισμού: x → y → z
x = torch.tensor([2.0, 3.0], requires_grad=True)  # παρακολούθησε τις πράξεις στο x
y = x ** 2        # y = [4.0, 9.0]
z = y.sum()       # z = 13.0

# Υπολόγισε τις κλίσεις αυτόματα
z.backward()      # «Πώς αλλάζει το z αν αλλάξω το x;»
print('Κλίσεις:', x.grad)  # tensor([4., 6.])

# Τι συνέβη μόλις τώρα;
# z = (x²).sum() → dz/dx = 2x
# Για x=[2, 3], η κλίση είναι 2*[2, 3] = [4, 6]
# Η backward() υπολόγισε την κλίση διατρέχοντας τον γράφο από το τέλος προς την αρχή!

print('\n✅ Έλεγχος με το χέρι: dz/dx = 2x')
print(f'   Για x=[2, 3]: 2*x = {2 * x.detach()}')

# Πότε να σταματήσεις την παρακολούθηση (για εξοικονόμηση μνήμης όταν κάνεις προβλέψεις):
print('\n--- Διακοπή παρακολούθησης κλίσεων ---')

# Επιλογή 1: Διαχειριστής συμφραζομένων (για ένα τμήμα κώδικα)
with torch.no_grad():
    y_no_grad = x * 2  # χωρίς παρακολούθηση κλίσεων
    print(f'Αποτέλεσμα χωρίς παρακολούθηση κλίσεων: {y_no_grad}')

# Επιλογή 2: Αποσύνδεση (για έναν μόνο τανυστή)
detached = x.detach()  # νέος τανυστής χωρίς ιστορικό πράξεων για την οπισθοδιάδοση
print(f'Αποσυνδεδεμένος τανυστής: {detached}')

## Σύνοψη

Έχεις πλέον δει όλες τις βασικές λειτουργίες του PyTorch για την ανάπτυξη μεγάλων γλωσσικών μοντέλων (LLM):

✅ **Βασικά στοιχεία των τανυστών** - δημιουργία, τύποι δεδομένων, συσκευές
✅ **Αναπαραγωγιμότητα** - αρχικές τιμές για τις γεννήτριες τυχαίων αριθμών κατά την αποσφαλμάτωση
✅ **Προσθήκη διαστάσεων** - σταδιακή μετάβαση 1D → 2D → 3D → 4D
✅ **Αλλαγή σχήματος και χρήση δεικτών** - πλοήγηση σε πολυδιάστατα δεδομένα
✅ **Μηχανισμός προσοχής** - υλοποίηση με το χέρι και καθιερωμένες πρακτικές για χρήση στην παραγωγή
✅ **Αιτιακή μάσκα** - εμποδίζει το μοντέλο να βλέπει τα επόμενα token
✅ **Διανυσματικές αναπαραστάσεις** - αναπαραστάσεις των token και των θέσεών τους
✅ **Broadcasting** - αυτόματη επέκταση σχημάτων
✅ **Μαθηματικές πράξεις** - πράξεις ανά στοιχείο, πολλαπλασιασμός πινάκων, αναγωγές
✅ **Autograd** - αυτόματη παραγώγιση
✅ **Βρόχοι εκπαίδευσης** - πρόσθιο πέρασμα, οπισθοδιάδοση και βελτιστοποίηση με περιορισμό των κλίσεων

**Επόμενα βήματα:** Στο Κεφάλαιο 7 θα δεις πώς να προετοιμάζεις πραγματικά κείμενα για εκπαίδευση!